In [25]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PANDAS DISPLAY SETTINGS (avoid line wrapping)
# --------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)


# --------------------------------------------------
# 1. LOAD ALL MATCH EVENT FILES
# --------------------------------------------------

folder = Path(r"Datasets\SkillCorner Premier League 24-25 data\dynamic_events_pl_24\dynamic")

dfs = []

for file in folder.glob("*.parquet"):
    df = pd.read_parquet(file)
    dfs.append(df)

events = pd.concat(dfs, ignore_index=True)

print("Total events loaded:", len(events))
print("Unique matches:", events["match_id"].nunique())


# --------------------------------------------------
# 2. FILTER PASSING OPTIONS
# --------------------------------------------------

passing_options = events[
    events["event_type"] == "passing_option"
].copy()

print("\nPassing option rows:", len(passing_options))


# --------------------------------------------------
# 3. KEEP IMPORTANT COLUMNS
# --------------------------------------------------

passing_options = passing_options[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "player_targeted_xpass_completion",
        "targeted"
    ]
].dropna(subset=["passing_option_score"])


# clean numeric columns
passing_options["xthreat"] = pd.to_numeric(
    passing_options["xthreat"], errors="coerce"
).fillna(0)

passing_options["player_targeted_xpass_completion"] = pd.to_numeric(
    passing_options["player_targeted_xpass_completion"], errors="coerce"
).fillna(1)


print("\nRows after cleaning:", len(passing_options))


# --------------------------------------------------
# 4. COUNT OPTIONS PER DECISION
# --------------------------------------------------

option_counts = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .size()
    .reset_index(name="n_options")
)

print("\nDecision moments:", len(option_counts))
print("Average options per decision:", option_counts["n_options"].mean())
print("Max options:", option_counts["n_options"].max())

print("\nDistribution of options per decision:")
print(option_counts["n_options"].value_counts().sort_index().to_string())


# --------------------------------------------------
# 5. REMOVE TRIVIAL DECISIONS
# --------------------------------------------------

option_counts = option_counts[option_counts["n_options"] >= 2]

passing_options = passing_options.merge(
    option_counts[["match_id", "associated_player_possession_event_id"]],
    on=["match_id", "associated_player_possession_event_id"]
)

print("\nPassing options after filtering trivial decisions:", len(passing_options))


# --------------------------------------------------
# 6. PASS VALUE (SkillCorner formula)
# --------------------------------------------------

passing_options["pass_value"] = (
    passing_options["passing_option_score"]
    * passing_options["xthreat"]
    * passing_options["player_targeted_xpass_completion"]
)


# --------------------------------------------------
# 7. BEST OPTION PER DECISION
# --------------------------------------------------

best_option = (
    passing_options
    .groupby(["match_id", "associated_player_possession_event_id"])
    .agg(
        best_option_score=("passing_option_score", "max"),
        best_xThreat_available=("xthreat", "max"),
        best_pass_value=("pass_value", "max"),
        avg_option_quality=("passing_option_score", "mean"),
        avg_option_xThreat=("xthreat", "mean")
    )
    .reset_index()
)

print("\nExample best options:")
print(best_option.head().to_string(index=False))


# --------------------------------------------------
# 8. IDENTIFY CHOSEN PASSES
# --------------------------------------------------

chosen = passing_options[
    passing_options["targeted"] == True
].copy()

chosen = chosen[
    [
        "match_id",
        "associated_player_possession_event_id",
        "player_id",
        "player_name",
        "passing_option_score",
        "xthreat",
        "player_targeted_xpass_completion",
        "pass_value"
    ]
]

chosen = chosen.rename(columns={
    "passing_option_score": "chosen_option_score",
    "xthreat": "chosen_xThreat",
    "pass_value": "chosen_pass_value"
})

print("\nExecuted passes:", len(chosen))
print("Example chosen passes:")
print(chosen.head().to_string(index=False))


# --------------------------------------------------
# 9. MERGE DECISION DATA
# --------------------------------------------------

decisions = chosen.merge(
    best_option,
    on=["match_id", "associated_player_possession_event_id"]
)

print("\nMerged decision rows:", len(decisions))


# --------------------------------------------------
# 10. COMPUTE DECISION METRICS
# --------------------------------------------------

decisions["decision_quality"] = (
    decisions["chosen_option_score"] /
    decisions["best_option_score"]
)

decisions["value_decision_quality"] = (
    decisions["chosen_pass_value"] /
    decisions["best_pass_value"]
)

decisions["creativity"] = (
    decisions["chosen_xThreat"] /
    decisions["best_xThreat_available"]
)

decisions["risk_taken"] = (
    decisions["best_option_score"] -
    decisions["chosen_option_score"]
)

decisions["xThreat_gain"] = (
    decisions["chosen_xThreat"] -
    decisions["best_xThreat_available"]
)

print("\nDecision quality summary:")
print(decisions["decision_quality"].describe())


# --------------------------------------------------
# 11. AGGREGATE PLAYER METRICS
# --------------------------------------------------

players = (
    decisions
    .groupby(["player_id", "player_name"])
    .agg(
        passes=("decision_quality", "count"),
        decision_quality=("decision_quality", "mean"),
        value_decision_quality=("value_decision_quality", "mean"),
        creativity=("creativity", "mean"),
        avg_xThreat_created=("chosen_xThreat", "mean"),
        avg_xThreat_gain=("xThreat_gain", "mean"),
        total_xThreat_gain=("xThreat_gain", "sum"),
        avg_option_quality=("avg_option_quality", "mean"),
        risk_taken=("risk_taken", "mean")
    )
    .reset_index()
)

print("\nPlayers before filtering:", len(players))

players = players[players["passes"] > 200]

print("Players with 200+ decisions:", len(players))


# --------------------------------------------------
# 12. BEST DECISION MAKERS
# --------------------------------------------------

print("\nBest decision makers (model optimal):")
print(
    players
    .sort_values("decision_quality", ascending=False)
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 13. MOST CREATIVE PASSERS
# --------------------------------------------------

print("\nMost creative passers:")
print(
    players
    .sort_values("creativity", ascending=False)
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 14. HIGHEST xTHREAT CREATORS
# --------------------------------------------------

print("\nHighest xThreat creators:")
print(
    players
    .sort_values("avg_xThreat_created", ascending=False)
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 15. BEST VALUE DECISION MAKERS
# --------------------------------------------------

print("\nBest value decision makers:")
print(
    players
    .sort_values("value_decision_quality", ascending=False)
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 16. HARDEST PASSING CONTEXT
# --------------------------------------------------

print("\nPlayers with hardest passing options:")
print(
    players
    .sort_values("avg_option_quality")
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 17. MOST xTHREAT GENERATED
# --------------------------------------------------

print("\nPlayers generating the most xThreat with passes:")
print(
    players
    .sort_values("total_xThreat_gain", ascending=False)
    .head(20)
    .to_string(index=False)
)

Total events loaded: 1811078
Unique matches: 378

Passing option rows: 939059

Rows after cleaning: 939058

Decision moments: 330426
Average options per decision: 2.8419615889790757
Max options: 35

Distribution of options per decision:
n_options
1      45304
2     106656
3      93067
4      49398
5      21731
6       8631
7       3478
8       1304
9        488
10       213
11        76
12        37
13        24
14         8
15         6
16         1
17         1
20         1
26         1
35         1

Passing options after filtering trivial decisions: 893754

Example best options:
 match_id associated_player_possession_event_id  best_option_score  best_xThreat_available  best_pass_value  avg_option_quality  avg_option_xThreat
  1650385                                   8_1             0.9321                  0.0021         0.001542            0.791675            0.000875
  1650385                                  8_10             0.9166                  0.0027         0.002017        

In [29]:
import pandas as pd

# ==============================
# LOAD DATA
# ==============================

path = r"Datasets/SkillCorner Premier League 24-25 data/players_match.parquet"

players_match = pd.read_parquet(path)

print("Rows:", len(players_match))


# ==============================
# CREATE PLAYER IDENTIFIERS
# ==============================

players_match["player_id"] = players_match["id"]
players_match["player_name"] = players_match["short_name"]
players_match["position"] = players_match["player_role_acronym"]
players_match["position_group"] = players_match["player_role_position_group"]


# ==============================
# MOST COMMON POSITION
# ==============================

position_counts = (
    players_match
    .groupby(["player_id","player_name","team_id","position","position_group"])
    .size()
    .reset_index(name="matches_at_position")
)

player_main_position = (
    position_counts
    .sort_values("matches_at_position", ascending=False)
    .drop_duplicates("player_id")
)

player_main_position = player_main_position[
    ["player_id","player_name","team_id","position","position_group"]
].rename(columns={"position":"main_position"})


# ==============================
# PLAYING TIME
# ==============================

minutes_df = (
    players_match
    .groupby("player_id")
    .agg(
        minutes_played_total=("playing_time_total_minutes_played","sum"),
        minutes_regular_time=("playing_time_total_minutes_played_regular_time","sum"),
        minutes_tip=("playing_time_total_minutes_tip","sum"),
        minutes_otip=("playing_time_total_minutes_otip","sum")
    )
    .reset_index()
)


# ==============================
# CREATE PER 90 DENOMINATORS
# ==============================

minutes_df["per90_factor_total"] = minutes_df["minutes_played_total"] / 90
minutes_df["per90_factor_regular"] = minutes_df["minutes_regular_time"] / 90


# ==============================
# MERGE POSITION + PLAYING TIME
# ==============================

player_info = player_main_position.merge(
    minutes_df,
    on="player_id",
    how="left"
)

print("\nExample player info:")
print(player_info.head())

Rows: 15176

Example player info:
   player_id   player_name  team_id main_position  position_group  minutes_played_total  minutes_regular_time  minutes_tip  minutes_otip  per90_factor_total  per90_factor_regular
0      18544    David Raya        3            GK           Other               3795.00               3795.00      1174.02        906.96           42.166667             42.166667
1      18526  D. Henderson       60            GK           Other               3770.73               3770.73       859.16       1175.54           41.897000             41.897000
2       1329       B. Leno       48            GK           Other               3796.68               3796.68      1093.70       1013.13           42.185333             42.185333
3      18948    O. Watkins       39            CF  Center Forward               2871.53               2871.53       796.94        717.89           31.905889             31.905889
4       7285       M. Sels      747            GK           Other      

In [31]:
import pandas as pd

# -------------------------------------------------------
# 1. LOAD PLAYERS MATCH DATA
# -------------------------------------------------------

players_match = pd.read_parquet(
    r"Datasets\SkillCorner Premier League 24-25 data\players_match.parquet"
)

print(players_match.columns)

# Typical useful columns
# player_id
# player_name
# team_name
# position


# -------------------------------------------------------
# 2. FIND MOST COMMON POSITION PER PLAYER
# -------------------------------------------------------

player_position = (
    players_match
    .groupby("player_id")["position"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={"position": "main_position"})
)


# -------------------------------------------------------
# 3. FIND PLAYER TEAM (most common team)
# -------------------------------------------------------

player_team = (
    players_match
    .groupby("player_id")["team_name"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)


# -------------------------------------------------------
# 4. MERGE POSITION + TEAM
# -------------------------------------------------------

player_info = player_position.merge(
    player_team,
    on="player_id",
    how="left"
)


# -------------------------------------------------------
# 5. MERGE INTO YOUR PLAYER RESULTS TABLE
# -------------------------------------------------------

players_summary = players_summary.merge(
    player_info,
    on="player_id",
    how="left"
)


# -------------------------------------------------------
# 6. REORDER COLUMNS (optional)
# -------------------------------------------------------

cols = [
    "player_id",
    "player_name",
    "team_name",
    "main_position",
    "passes",
    "decision_quality",
    "value_decision_quality",
    "creativity",
    "avg_xThreat_created",
    "avg_xThreat_gain",
    "total_xThreat_gain",
    "avg_option_quality",
    "risk_taken"
]

players_summary = players_summary[cols]


# -------------------------------------------------------
# 7. CHECK RESULT
# -------------------------------------------------------

print(players_summary.head())

Index(['start_time', 'end_time', 'number', 'yellow_card', 'red_card', 'injured', 'goal', 'own_goal', 'team_player_id', 'team_id', 'id', 'first_name', 'last_name', 'short_name', 'birthday', 'trackable_object', 'gender', 'player_role_id', 'player_role_position_group', 'player_role_name', 'player_role_acronym', 'playing_time_total_minutes_tip', 'playing_time_total_minutes_otip', 'playing_time_total_start_frame', 'playing_time_total_end_frame', 'playing_time_total_minutes_played', 'playing_time_total_minutes_played_regular_time', 'playing_time_by_period', 'playing_time_total', 'match_id'], dtype='object')


KeyError: 'player_id'